# Báo cáo Project
Lớp TTNT-162297, Nhóm G08

## 1. Thông tin chung

> Thêm khối trích dẫn



### Thành viên
- Nguyễn Thái An 202416110
- Trần Bình Minh 202416294
- Nguyễn Thanh Hải 202416192
- Bùi Hoàng Tú 202416378

### Lịch thực hiện
- W2: Đăng ký nhóm
- W3-4: Đề xuất project (03/10)
- W08: Báo cáo tiến độ giữa kỳ (24/10)
- W15: Hoàn thành và gửi báo cáo project (12/12)
- W16-18: Trình bày project, Q&A

## 2. Đề xuất project (W3)

### Bài toán
Nghiên cứu và tối ưu hóa bài toán giải và sinh hình thế Sudoku dạng CSP (Constraint Satisfaction Problem). Xây dựng giải pháp cho phép nhận diện tính hợp lệ của cấu hình người dùng đánh trước vài ô ngẫu nhiên và tự động sinh ra các hình thế hoàn chỉnh. Đánh giá thuật toán bằng nhiều metric và giao diện trực quan.

### Phương pháp dự kiến
Xây dựng mô hình 4 cấp độ: Naive Backtracking $\to$ Forward Checking $\to$ MRV/Degree Heuristic $\to$ AC-3 kết hợp Randomized Candidate Selection.

### Phân công
- Nguyễn Thái An: Phân tích các thuật toán và triển khai
- Trần Bình Minh: Đánh giá các thuật toán theo các metrics (chưa rõ các chỉ số)
- Nguyễn Thanh Hải: Phân tích các thuật toán và triển khai
- Bùi Hoàng Tú: Dashboard & Trực quan hóa Tương tác

## 3. Tiến độ giữa kỳ (W8)

### Phương pháp hiện tại
So với đề xuất ban đầu ở tuần 3 (xây dựng tuần tự 4 cấp độ cố định: Naive Backtracking $\to$ Forward Checking $\to$ MRV/Degree $\to$ AC-3 kết hợp Randomized Candidate Selection, kèm bài toán sinh đề), hiện tại nhóm điều chỉnh theo hướng thiết kế mô-đun trên cùng một bộ khung CSP. Lý do là các kỹ thuật trong bài giảng thực chất giải quyết những khâu khác nhau của quá trình tìm kiếm: chọn biến, chọn giá trị, lan truyền ràng buộc và xử lý khi gặp ngõ cụt. Nếu tách cứng thành 4 mức thì rất khó bật/tắt từng thành phần để quan sát vai trò riêng của mỗi kỹ thuật. Ngoài ra, thay vì đưa yếu tố ngẫu nhiên vào bước chọn ứng viên của DFS, nhóm tách hẳn một nhánh tìm kiếm cục bộ (Local Search - Min-Conflicts) để đối chiếu với tìm kiếm hệ thống theo chiều sâu. Ở giai đoạn giữa kỳ, chương trình tập trung vào kiểm tra tính hợp lệ của cấu hình đầu vào, giải Sudoku 9x9 và trực quan hóa từng bước; phần tự động sinh hình thế hoàn chỉnh và kiểm tra nghiệm duy nhất chưa triển khai.

**Mô hình hóa bài toán theo CSP và cách biểu diễn ràng buộc nhị phân:**
- *Biến và miền giá trị:* Bàn Sudoku $9\times 9$ gồm $81$ biến rời rạc $X_{(r,c)}$ với $r, c \in \{0..8\}$. Các ô cho trước của đề bài được đưa vào tập `locked_cells` và có miền đơn trị $D(X_{(r,c)}) = [v_0]$; các ô trống có miền ban đầu $D(X_{(r,c)}) \subseteq \{1..9\}$ sau khi lọc bỏ các giá trị trùng trực tiếp với đề bài.
- *Chuyển về ràng buộc nhị phân (Binary Constraints):* Về mặt toán học, mỗi hàng, mỗi cột và mỗi khối $3\times 3$ đều mang một ràng buộc toàn cục $\text{AllDifferent}(X_1, \dots, X_9)$. Trong chương trình, mỗi ràng buộc 9 ngôi này được phân rã tương đương thành $\binom{9}{2} = 36$ ràng buộc nhị phân $X_i \neq X_j$. Hai ô bất kỳ $(r,c)$ và $(a,b)$ (với $(a,b) \neq (r,c)$) có ràng buộc nhị phân khi và chỉ khi thỏa ít nhất một trong ba điều kiện: cùng hàng ($a = r$), cùng cột ($b = c$), hoặc cùng khối $3\times 3$ ($\lfloor a/3 \rfloor = \lfloor r/3 \rfloor$ và $\lfloor b/3 \rfloor = \lfloor c/3 \rfloor$).
- *Cấu trúc dữ liệu đồ thị ràng buộc (`peers`):* Khi khởi tạo bộ giải, chương trình dựng sẵn từ điển kề `self.peers[(r, c)]` lưu tập hợp (`set`) tất cả các ô có ràng buộc nhị phân với $(r,c)$. Nhờ dùng `set`, các ô vừa cùng hàng/cột vừa cùng khối $3\times 3$ được tự động khử trùng lặp, nên mỗi biến có chính xác $8 + 8 + 4 = 20$ biến láng giềng (`peers`). Toàn bộ bàn cờ tạo thành đồ thị ràng buộc 20-chính quy gồm $81$ đỉnh, $\frac{81 \times 20}{2} = 810$ cạnh vô hướng $X_i \neq X_j$ (tương ứng $1620$ cung có hướng $(X_i \to X_j)$ khi xét lan truyền AC-3). Cách biểu diễn này bám sát đồ thị ràng buộc trong bài giảng và cho phép cài đặt thống nhất việc kiểm tra hợp lệ `is_safe`, thu hẹp miền giá trị cũng như đếm xung đột.

**Các kỹ thuật đang sử dụng, lý do lựa chọn, ưu/nhược điểm và thủ thuật triển khai:**

1. **Backtracking Search (DFS - Tìm kiếm quay lui):**
   - *Lý do sử dụng:* Thay vì sinh toàn bộ bảng rồi mới kiểm tra (Generate and Test) dẫn tới số trạng thái $O(9^n)$ quá lớn, Backtracking gán lần lượt từng ô và kiểm tra ràng buộc ngay sau mỗi phép gán để cắt sớm các nhánh sai. Đây là khung tìm kiếm nền tảng cho các kỹ thuật cải tiến phía sau.
   - *Ưu điểm:* Đảm bảo tính đầy đủ và đúng đắn trên miền hữu hạn (nếu đề có nghiệm và đủ thời gian thì tìm được nghiệm; nếu duyệt hết không gian mà không có lời giải thì kết luận được đề vô nghiệm), đồng thời dễ kết hợp với heuristics và inference.
   - *Nhược điểm:* Nếu chỉ dùng Backtracking thuần (chọn ô tuần tự, không thu hẹp miền của các ô lân cận) thì phát hiện mâu thuẫn muộn, dễ lặp lại cùng một lỗi ở nhiều nhánh và số lần quay lui tăng rất nhanh trên các đề nhiều ô trống.
   - *Thủ thuật triển khai:* Thay vì viết nhiều hàm đệ quy rời rạc cho từng biến thể (`solve_naive`, `solve_fc`, `solve_mac`,...), nhóm thiết kế duy nhất một hàm đệ quy khung `backtrack()` theo mẫu **Template Method kết hợp cờ cấu hình (Flags)**. Hàm khung chia rõ 4 bước chuẩn: chọn biến $\to$ sắp thứ tự giá trị $\to$ gán và chạy suy diễn $\to$ khôi phục trạng thái (`deepcopy` miền giá trị và tập nguyên nhân tại một điểm duy nhất trước khi thử số) khi quay lui.

2. **Heuristics chọn biến: MRV (Minimum Remaining Values) kết hợp Degree Heuristic:**
   - *Lý do sử dụng:* Trong quay lui, thứ tự chọn biến ảnh hưởng trực tiếp tới độ rộng của cây tìm kiếm. Nhóm dùng MRV để ưu tiên ô có ít giá trị hợp lệ nhất (biến bị ràng buộc chặt nhất); khi có nhiều ô cùng số lượng giá trị hợp lệ ít nhất thì dùng Degree Heuristic để ưu tiên ô có nhiều ô liên quan chưa gán nhất.
   - *Ưu điểm:* Áp dụng nguyên lý "fail-first" (phát hiện nhánh sai càng sớm càng tốt), giảm hệ số phân nhánh ngay từ các tầng trên nên thường giảm mạnh số phép gán và số lần quay lui so với chọn ô theo thứ tự hàng/cột cố định.
   - *Nhược điểm:* Mỗi bước đệ quy phải quét các ô trống và tính số giá trị hợp lệ/số láng giềng chưa gán. Với những đề rất dễ hoặc khi không kết hợp cắt miền, chi phí tính heuristic ở mỗi nút vẫn phát sinh đều đặn.
   - *Thủ thuật triển khai:* Gom cả MRV, Degree và thứ tự phá hòa cuối cùng vào một lần gọi `min(empty, key=...)` trả về bộ khóa `(len(available_values(cell)), -unassigned_peers, cell)` (dùng dấu âm cho bậc tự do để tìm cực đại ngay trong hàm `min`). Hàm `available_values(cell)` lọc lại `domains[cell]` qua `is_safe` để MRV vẫn đếm đúng số ứng viên hợp lệ ngay cả khi tắt lan truyền ràng buộc (`inference = none`).

3. **Heuristic chọn giá trị: LCV (Least Constraining Value):**
   - *Lý do sử dụng:* Sau khi đã chọn được một ô để gán, thứ tự thử các số trong miền của ô đó quyết định khả năng đi tới nghiệm sớm hay muộn. LCV xếp các giá trị sao cho ưu tiên thử trước giá trị làm mất ít lựa chọn của các ô liên quan chưa gán nhất.
   - *Ưu điểm:* Giữ lại nhiều sự linh hoạt nhất cho các biến kế tiếp, tăng xác suất đi trúng nhánh có nghiệm sớm khi mục tiêu là tìm một lời giải.
   - *Nhược điểm:* Tốn thêm chi phí kiểm tra miền của các ô láng giềng để chấm điểm và sắp xếp giá trị. Nếu nhánh hiện tại thực chất là ngõ cụt (bắt buộc phải thử hết mọi giá trị của biến rồi mới quay lui) thì LCV không giúp giảm số nhánh phải duyệt mà còn làm tăng thời gian xử lý tại nút đó.
   - *Thủ thuật triển khai:* Không cần giả lập gán rồi sao chép miền giá trị cho từng ứng viên; chương trình chấm điểm nhanh mỗi giá trị `val` bằng cách đếm trực tiếp số ô láng giềng chưa gán trong `peers[(r, c)]` hiện đang chứa `val` trong tập hợp lệ của chúng, rồi `sorted` tăng dần theo bộ khóa `(impact_count, val)`.

4. **Forward Checking (FC - Kiểm tra tiến):**
   - *Lý do sử dụng:* Khắc phục nhược điểm phát hiện lỗi muộn của Backtracking thuần. Ngay khi gán $X = v$, FC xóa $v$ khỏi miền giá trị của mọi ô liên quan chưa gán; nếu có ô nào bị rỗng miền thì dừng nhánh đó ngay lập tức.
   - *Ưu điểm:* Chi phí mỗi bước thấp vì chỉ kiểm tra các ô kề trực tiếp với biến vừa gán, phát hiện sớm các thất bại trực tiếp và cung cấp miền giá trị cập nhật hơn cho MRV/LCV.
   - *Nhược điểm:* Chỉ đảm bảo tính phù hợp cục bộ giữa biến vừa gán với láng giềng trực tiếp của nó, chưa nhìn xa hơn. Khi miền của một láng giềng bị thu hẹp còn một giá trị duy nhất, FC chưa tự lan truyền ràng buộc đó sang các ô tiếp theo nên vẫn bỏ sót các mâu thuẫn gián tiếp.
   - *Thủ thuật triển khai:* Khi gán $(r,c) = val$, hàm `forward_check` chỉ duyệt đúng 20 đỉnh trong `peers[(r, c)]` còn trống; nếu `val` nằm trong `domains[peer]` thì gọi `prune` xóa đi, ghi nhận nguyên nhân cắt miền là `{(r, c)}` và dừng trả về `False` ngay tại thời điểm phát hiện bất kỳ `domains[peer]` nào bị rỗng (early exit).

5. **Arc Consistency (AC-3 / MAC - Duy trì phù hợp cung):**
   - *Lý do sử dụng:* Để phát hiện mâu thuẫn sớm hơn FC, nhóm cài AC-3 kiểm tra tính phù hợp cạnh $(X \to Y)$: với ràng buộc khác số trong Sudoku, nếu miền của $Y$ chỉ còn một giá trị duy nhất $v$ thì $v$ bị xóa khỏi miền của $X$, sau đó mọi cung $(Z \to X)$ liên quan được đưa lại vào hàng đợi để lan truyền tiếp. Khi gắn AC-3 vào trước tìm kiếm và sau mỗi phép gán trong DFS, bộ giải hoạt động theo cơ chế MAC (Maintaining Arc Consistency).
   - *Ưu điểm:* Cắt tỉa miền giá trị sâu hơn FC qua nhiều bước lan truyền dây chuyền, phát hiện miền rỗng sớm hơn rất nhiều trên các cấu hình phức tạp và giảm đáng kể số nhánh vô ích.
   - *Nhược điểm:* Chi phí duy trì hàng đợi cung và sao lưu/khôi phục miền giá trị lớn hơn FC. Trên các đề đơn giản hoặc khi FC + MRV đã đủ dẫn thẳng tới nghiệm, thời gian chạy thực tế của AC-3 có thể cao hơn do chi phí kiểm tra cung ở mỗi bước.
   - *Thủ thuật triển khai:* Tối ưu hóa hàm `revise(X, Y)` chuyên biệt cho ràng buộc bất đẳng thức $X \neq Y$: thay vì duyệt cặp giá trị $O(|D(X)| \times |D(Y)|)$ tổng quát, một giá trị trong $D(X)$ chỉ mất giá trị hỗ trợ ở $D(Y)$ khi và chỉ khi $|D(Y)| == 1$. Nhờ đó `revise` rút xuống kiểm tra $O(1)$ (`len(domains[y]) == 1`). Hàng đợi cung dùng `collections.deque`; trước khi vào DFS sẽ chạy AC-3 toàn cục trên 1620 cung, còn sau mỗi phép gán tại `cell` chỉ khởi tạo hàng đợi bằng 20 cung hướng vào `[(p, cell) for p in peers[cell]]`.

6. **Conflict-Directed Backjumping (CBJ - Quay lui có định hướng xung đột):**
   - *Lý do sử dụng:* Trong bài giảng có nêu hạn chế của quay lui chuẩn (Standard Backtracking) là chỉ lùi lại đúng biến liền trước nên dễ lặp lại lỗi khi biến đó không phải nguyên nhân gây mâu thuẫn. Nhóm bổ sung CBJ bằng cách lưu tập nguyên nhân cắt miền cho từng ô để khi thất bại có thể bỏ qua các tầng quyết định không liên quan.
   - *Ưu điểm:* Tránh việc thử vô ích tất cả giá trị của những biến trung gian không góp phần tạo ra mâu thuẫn ở tầng sâu, hỗ trợ thoát khỏi vùng tìm kiếm sai nhanh hơn trong các cây đệ quy sâu.
   - *Nhược điểm:* Phải lưu trữ, hợp nhất và khôi phục tập nguyên nhân cùng với miền giá trị nên tốn thêm bộ nhớ và thao tác xử lý. Khi đã dùng MRV và AC-3 mạnh, nhiều mâu thuẫn được phát hiện sớm ngay gần nơi phát sinh nên lợi ích nhảy tầng của CBJ không phải lúc nào cũng vượt qua chi phí theo dõi nguyên nhân.
   - *Thủ thuật triển khai:* Duy trì từ điển `reasons[cell]` lưu tập các ô quyết định đã trực tiếp hoặc gián tiếp (qua chuỗi lan truyền FC/AC-3) cắt bớt miền của `cell`. Hàm `backtrack()` trả về cặp `(solved, conflict_set)`: nếu một nhánh thất bại và `cell` hiện tại không nằm trong `causes` trả về từ tầng dưới (`cell not in causes`), hàm lập tức hoàn tác ô hiện tại rồi trả thẳng `(False, causes)` lên tầng cha mà không thử tiếp các giá trị còn lại của `cell` (hiện thực hóa bước nhảy cóc ngay trên ngăn xếp đệ quy).

7. **Local Search với chiến lược Min-Conflicts (Tìm kiếm cục bộ):**
   - *Lý do sử dụng:* Theo phần cuối của bài giảng CSP, thay vì xây dựng lời giải từng biến từ bảng rỗng, có thể làm việc trên một phép gán đầy đủ cho toàn bộ các biến rồi sửa dần các vi phạm ràng buộc. Nhóm điền ngẫu nhiên các ô trống, sau đó lặp: chọn ngẫu nhiên một ô đang có xung đột và đổi sang giá trị làm số ràng buộc bị vi phạm ít nhất; nếu bị kẹt trên plateau sau một số bước không cải thiện thì khởi tạo lại bảng.
   - *Ưu điểm:* Không phải duy trì cây đệ quy hay các bản sao miền giá trị theo độ sâu, thao tác mỗi bước đơn giản và cho góc nhìn trực quan rất khác so với DFS.
   - *Nhược điểm:* Không có tính đầy đủ: thuật toán dễ rơi vào cực tiểu cục bộ hoặc đi ngang trên plateau nên hết ngân sách bước lặp mà chưa tìm ra lời giải thì chỉ kết luận là chưa tìm thấy (`not_found`), không chứng minh được đề vô nghiệm. Với Sudoku 9x9 có ràng buộc chặt giữa hàng, cột và khối, Min-Conflicts thuần thường kém ổn định hơn DFS có suy diễn.
   - *Thủ thuật triển khai:* Khi thử đổi giá trị của một ô `cell`, các cạnh ràng buộc không nối với `cell` giữ nguyên trạng thái, nên chương trình chỉ cần tính hàm mục tiêu cục bộ `conflict_count(cell, v)` trên 20 ô trong `peers[cell]` thay vì tính lại tổng lỗi toàn bảng 9 lần. Các giá trị đồng hạng ít xung đột nhất được chọn ngẫu nhiên (`random.choice`) để thoát plateau nhỏ, kết hợp cơ chế **Random Restart** (khởi tạo lại các ô không khóa khi trải qua 500 bước liên tiếp không giảm được kỷ lục lỗi `best`).

### Chương trình
Chương trình giải Sudoku và giao diện trực quan đã được hoàn thiện. Mã nguồn của nhóm: https://github.com/anthaimagic213/BTL-IntroAI-G8-GV-Do_Tuan_Dung.git

### Phân tích kết quả, vấn đề gặp phải
Chương trình hiện đã hoàn thiện phần giải Sudoku và hiển thị các bước chạy của thuật toán. Tuy nhiên, nhóm chưa có số liệu metrics cụ thể và biểu đồ để đánh giá, so sánh các thuật toán trên những dữ liệu đầu vào khác nhau. Vì vậy, hiện chưa giải thích rõ được với dạng đề nào kỹ thuật này chạy nhanh hơn kỹ thuật kia, hoặc vì sao lượng bộ nhớ sử dụng lại nhiều hơn hay ít hơn. Nhóm sẽ bổ sung phần đo đạc và trực quan hóa kết quả ở giai đoạn tiếp theo.

## 4. Cập nhật kết quả cuối kỳ (W15)

### Chi tiết phương pháp, dữ liệu
....

### Chương trình
...

### Phân tích, đánh giá kết quả
...

### Cập nhật phân công, khối lượng công việc
<!-- công việc của các thành viên, tỷ lệ đóng góp của các thành viên -->
...